In [9]:
import sys, os
sys.path.append(os.path.abspath(".."))

from baseline.generate import generate
import baseline.generate as gen_module
from baseline.generate import filter_invalid
from evaluation.evaluator import (
    extract_code, evaluate, get_defined_functions, get_called_functions,
    get_function_name, run_candidate, deduplicate_candidates,
    extract_call_args, get_probe_inputs, find_disagreements
)
from evaluation.eig import (
    init_weights, compute_pt, eig_score, full_score, build_test_pool,
    should_ask, update_weights
)
from datasets import load_dataset
import json

dataset = load_dataset("google-research-datasets/mbpp")

In [3]:
with open("../results/phase2_ambiguity_7b.json", "r") as f:
    phase2_7b_30 = json.load(f)

ambiguous_tasks = [r for r in phase2_7b_30 if r.get("n_disagreement_points", 0) > 0]
print(f"{len(ambiguous_tasks)} ambiguous tasks available")
for r in ambiguous_tasks:
    print(f"  task {r['task_id']}: {r['n_disagreement_points']} disagreement points")

15 ambiguous tasks available
  task 602: 1 disagreement points
  task 603: 3 disagreement points
  task 607: 3 disagreement points
  task 608: 3 disagreement points
  task 609: 3 disagreement points
  task 610: 3 disagreement points
  task 612: 3 disagreement points
  task 617: 3 disagreement points
  task 619: 3 disagreement points
  task 622: 3 disagreement points
  task 623: 3 disagreement points
  task 625: 3 disagreement points
  task 626: 3 disagreement points
  task 627: 3 disagreement points
  task 630: 3 disagreement points


### Trying Out a Human Oracle

In [4]:
def human_oracle_answer(func_name, test_input, hypothesized_output_repr, run_fn):
    print(f"\nClarification question:")
    print(f"  Does the function return {hypothesized_output_repr} for input {test_input}?")

    while True:
        answer = input("Your answer (y/n): ").strip().lower()
        if answer in ("y", "yes"):
            return 1
        elif answer in ("n", "no"):
            return 0
        else:
            print("Please type y or n.")

In [5]:
def clarification_loop_human(sample, candidates, k_max=4, epsilon=0.02, gamma=0.85):
    func_name = list(get_called_functions(sample["test_list"]))[0]
    weights = init_weights(candidates)

    probe_inputs = get_probe_inputs(sample["test_list"])
    disagreements = find_disagreements(candidates, probe_inputs, func_name)
    test_pool = build_test_pool(disagreements)

    questions_asked = []

    print(f"\n{'='*60}")
    print(f"Task {sample.get('task_id')}: {sample['text']}")
    print(f"{'='*60}")

    for _ in range(k_max):
        if not test_pool:
            break

        scored = []
        for (inp, hyp_output) in test_pool:
            pt, outcomes, defined_w = compute_pt(candidates, weights, run_candidate, func_name, inp, hyp_output)
            undefined_w = 1 - defined_w
            s = full_score(pt, defined_w, undefined_w, epsilon)
            scored.append((s, inp, hyp_output, outcomes))

        scored.sort(key=lambda x: -x[0])
        best_score, best_input, best_output, best_outcomes = scored[0]

        if not should_ask(candidates, weights, run_candidate, func_name, best_input, best_output, gamma):
            print("\n(Model confident enough — stopping without asking.)")
            break

        answer = human_oracle_answer(func_name, best_input, best_output, run_candidate)

        questions_asked.append({
            "input": best_input, "hypothesized_output": best_output,
            "answer": answer, "score": best_score
        })

        weights = update_weights(weights, best_outcomes, answer, epsilon)
        test_pool.remove((best_input, best_output))

    best_idx = max(weights, key=weights.get)
    print(f"\nFinal answer chosen:\n{candidates[best_idx]}")

    return {
        "task_id": sample.get("task_id"),
        "n_questions_asked": len(questions_asked),
        "questions": questions_asked,
        "final_candidate": candidates[best_idx],
        "final_weights": weights
    }

In [ ]:
task_id_to_try = 608  # bell_Number 

phase2_entry = next(r for r in ambiguous_tasks if r["task_id"] == task_id_to_try)
sample = next(s for s in dataset["train"] if s["task_id"] == task_id_to_try)

result_608 = clarification_loop_human(sample, phase2_entry["candidates"], k_max=4)

passed, error = evaluate(
    f"```python\n{result_608['final_candidate']}\n```",
    sample["test_list"],
    setup_code=sample.get("test_setup_code", "")
)
print(f"\nActually correct against hidden tests? {passed}")


Task 608: Write a python function to find nth bell number.

Clarification question:
  Does the function return 0 for input (2,)?

(Model confident enough — stopping without asking.)

Final answer chosen:
def bell_Number(n): 
    Bell = [[0 for x in range(n+1)] for y in range(n+1)] 
    Bell[0][0] = 1
    for i in range(1,n + 1):  
        Bell[i][0] = Bell[i-1][i-1]  
        for j in range(1,i + 1):  
            Bell[i][j] = Bell[i-1][j-1] + Bell[i][j-1]   
    return Bell[n][0]

Actually correct against hidden tests? True


### Trying Human Oracle on Custom Prompts - Just for the fun of it

In [8]:
custom_sample = {
    "task_id": "custom_1",
    "text": "Write a function to remove duplicate names from a list, ignoring case.",
    "test_list": [
        "assert dedup_names(['Bob', 'bob', 'Alice']) == ['Bob', 'Alice']"
    ],
    "test_setup_code": ""
}

In [10]:
def generate_candidates(problem_text, test_example, n=8, temperature=0.8):
    """Generate N diverse candidate solutions for one problem."""
    candidates = []
    for i in range(n):
        response = generate(
            problem_text,
            test_example=test_example,
            temperature=temperature,
            do_sample=True
        )
        code = extract_code(response)
        candidates.append(code)
    return candidates


def analyze_ambiguity(sample, n_candidates=8, temperature=0.8):
    """
    Full Phase 2 pipeline for a single MBPP-shaped sample.
    """
    problem_text = sample["text"]
    test_list = sample["test_list"]
    expected_func_name = get_called_functions(test_list)
    expected_func_name = list(expected_func_name)[0] if expected_func_name else None

    if expected_func_name is None:
        return {"error": "could not determine expected function name"}

    raw_candidates = generate_candidates(
        problem_text,
        test_example=test_list[0],
        n=n_candidates,
        temperature=temperature
    )

    valid_candidates = filter_invalid(raw_candidates)

    probe_inputs = get_probe_inputs(test_list)

    if not probe_inputs:
        return {
            "task_id": sample.get("task_id"),
            "note": "no simple-literal probe inputs available (custom class args?)",
            "n_valid_candidates": len(valid_candidates)
        }

    unique_candidates = deduplicate_candidates(
        valid_candidates, probe_inputs, expected_func_name
    )

    disagreements = find_disagreements(
        unique_candidates, probe_inputs, expected_func_name
    )

    return {
        "task_id": sample.get("task_id"),
        "n_raw_candidates": len(raw_candidates),
        "n_valid_candidates": len(valid_candidates),
        "n_unique_candidates": len(unique_candidates),
        "n_disagreement_points": len(disagreements),
        "disagreements": disagreements,
        "candidates": unique_candidates
    }

In [11]:
result = analyze_ambiguity(custom_sample, n_candidates=8, temperature=0.8)
print(f"Unique candidates: {result.get('n_unique_candidates')}")
print(f"Disagreement points: {result.get('n_disagreement_points')}")

Unique candidates: 2
Disagreement points: 0


In [15]:
if result.get("n_disagreement_points", 0) > 0:
    clarif_result = clarification_loop_human(custom_sample, result["candidates"], k_max=4)
    final_code = clarif_result["final_candidate"]
else:
    print("No disagreement detected — model converged on one interpretation.")
    # Just take the first (only, since it converged) candidate directly
    final_code = result["candidates"][0]
    clarif_result = None  # no clarification actually happened

print(final_code)

No disagreement detected — model converged on one interpretation.
def dedup_names(names):
    seen = set()
    result = []
    for name in names:
        if name.lower() not in seen:
            seen.add(name.lower())
            result.append(name)
    return result


### Trying Out on a Codeforces Problem

In [17]:
codeforces_sample = {
    "task_id": "cf_1097b",
    "text": (
        "Given a list of n rotation angles (in degrees, each between 1 and 180), "
        "determine whether it is possible to choose a direction (clockwise or "
        "counterclockwise) for each rotation such that the total rotation sums "
        "to a multiple of 360 degrees. Return True if possible, False otherwise."
    ),
    "test_list": [
        "assert can_return_to_zero([10, 20, 30]) == True",
        
        "assert can_return_to_zero([10, 10, 10]) == False",
        "assert can_return_to_zero([120, 120, 120]) == True",
    ],
    "test_setup_code": ""
}

result = analyze_ambiguity(codeforces_sample, n_candidates=8, temperature=0.8)
print(f"Unique candidates: {result.get('n_unique_candidates')}")
print(f"Disagreement points: {result.get('n_disagreement_points')}")

Unique candidates: 3
Disagreement points: 2


In [18]:
if result.get("n_disagreement_points", 0) > 0:
    clarif_result = clarification_loop_human(codeforces_sample, result["candidates"], k_max=4)
    final_code = clarif_result["final_candidate"]
else:
    print("No disagreement — candidates converged.")
    final_code = result["candidates"][0]

print(final_code)


Task cf_1097b: Given a list of n rotation angles (in degrees, each between 1 and 180), determine whether it is possible to choose a direction (clockwise or counterclockwise) for each rotation such that the total rotation sums to a multiple of 360 degrees. Return True if possible, False otherwise.

Clarification question:
  Does the function return True for input ([10, 20, 30],)?

Clarification question:
  Does the function return False for input ([10, 10, 10],)?

(Model confident enough — stopping without asking.)

Final answer chosen:
def can_return_to_zero(angles):
    def dfs(index, total, direction):
        if index == len(angles):
            return total % 360 == 0
        
        if dfs(index + 1, total + angles[index], 'clockwise'):
            return True
        if dfs(index + 1, total - angles[index], 'counterclockwise'):
            return True
        
        return False
    
    return dfs(0, 0, None)
def can_return_to_zero(angles):
    def dfs(index, total, directio

In [20]:
my_hidden_tests = [
    "assert can_return_to_zero([180]) == False",   # neither +180 nor -180 alone hits 0 mod 360
    "assert can_return_to_zero([90, 90, 90, 90]) == True",  # all same direction = 360
]

passed, error = evaluate(f"```python\n{final_code}\n```", my_hidden_tests)
print(passed, error)

True None
